# 04 — 모듈 ① 전체 파이프라인 실행 (D-3 누수차단·재현성 반영)

`docs/reproducibility_pipeline.md`의 정석 순서를 Colab에서 한 번에 실행한다.

**핵심 (이전 파이프라인 대비 바뀐 점):**
- 합성 병합 시 **MinHash 중복 제거**로 시드↔합성 누수 차단
- Stacking은 **OOF logits**만 사용(in-fold 누수 제거) — `generate_oof_logits.py`
- 모든 run **MLflow 자동 로깅**(`mlruns/`)
- 지표는 긴급(3) 포함 **4-class 고정 채점**

**런타임 요건**: GPU (A100 권장). 2·3단계가 KcELECTRA fine-tune.

> ⚠️ **3단계(OOF) 비용 주의**: 5-fold면 base fine-tune을 **5회** 더 돈다(각 fold가 train의 4/5).
> A100에서도 OOF 생성만 수 시간 걸릴 수 있다. 비용이 부담되면 아래 `OOF_FOLDS=3`으로 낮춘다.

## 0. GPU 확인

In [ ]:
!nvidia-smi | head -15
import torch
print(f"\nPyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"Device: {p.name}, VRAM: {p.total_memory/1e9:.1f} GB")
else:
    print("⚠ GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU 선택.")

## 1. Repo Clone (private)

GitHub Personal Access Token 필요. `REPO_OWNER`를 본인 username으로 수정.
**재실행 안전**: 이미 clone돼 있으면 pull.

In [ ]:
import os, getpass, subprocess
from pathlib import Path

REPO_OWNER = "threeGuineas"   # ← 본인 GitHub username으로 수정
REPO_NAME = "thisabled-ai"
BRANCH = "fix/d3-leakage-reproducibility"   # D-3 작업 브랜치
REPO_DIR = Path(f"/content/{REPO_NAME}")

if REPO_DIR.exists():
    print(f"Repo exists at {REPO_DIR} — fetch & checkout {BRANCH}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT: ")
    url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO_DIR)], check=True)
    print(f"Cloned ({BRANCH}) to {REPO_DIR}")

%cd {REPO_DIR}
!git log --oneline -3

## 2. 의존성 설치

`requirements-colab.txt`에 `mlflow`·`datasketch`가 포함돼 있어야 D-3 단계가 동작한다.

In [ ]:
!pip install -q -r requirements-colab.txt
!pip install -q "accelerate>=0.26"
import importlib
for m in ("mlflow", "datasketch"):
    print(m, "OK" if importlib.util.find_spec(m) else "❌ 누락 — requirements-colab.txt 확인")

## 3. 파이프라인 파라미터

한 곳에서 노브를 설정하고 이후 셀에서 환경변수로 전달한다.

In [ ]:
import os
os.environ["CONFIG"]       = "configs/module1_kcelectra_ce.yaml"
os.environ["SYNTH_REPEAT"] = "8"     # 합성 oversample 배수 (0이면 합성 미포함)
os.environ["OOF_FOLDS"]    = "5"     # OOF Stratified fold 수 (비용 부담시 3)
os.environ["MODEL_DIR"]    = "models/checkpoints/module1_ce"  # config paths.checkpoint_dir와 일치
print({k: os.environ[k] for k in ("CONFIG","SYNTH_REPEAT","OOF_FOLDS","MODEL_DIR")})

### 3-1. 데이터 빌드 (시드 다운로드 → processed → 합성 병합 + MinHash 중복 제거)

`build_final_dataset.py`가 합성 train을 시드와 비교해 근사 중복(Jaccard≥0.8)을 제거한 뒤 oversample 한다.

In [ ]:
!python scripts/download_seed_datasets.py
!python scripts/build_processed_dataset.py
!python scripts/build_final_dataset.py --synth-repeat $SYNTH_REPEAT

### 3-2. base KcELECTRA fine-tune (MLflow 자동 로깅)

In [ ]:
!python scripts/train_module1.py --config $CONFIG

### 3-3. OOF logits 생성 (Stratified K-fold, seed=42)

⚠️ **여기가 비용 큰 단계** — fold 수만큼 base를 새로 fine-tune 한다.
산출: `data/processed/oof_train_logits.npy` (train.parquet 행과 정렬).

In [ ]:
!python scripts/generate_oof_logits.py --config $CONFIG --folds $OOF_FOLDS

### 3-4. Stacking 메타러너 (train=OOF, val/test=full-train base)

In [ ]:
!python scripts/train_stacking.py --config $CONFIG --model-dir $MODEL_DIR

### 3-5. 합성 hold-out 평가

⚠️ 이 Recall은 **합성→합성 순환** 평가다. 실데이터 긴급 일반화를 보장하지 않는다(D-2에서 실데이터 hold-out 구성 예정).

In [ ]:
!python scripts/evaluate_synthetic.py --model-dir $MODEL_DIR

> **대안 (한 번에 실행)**: 위 3-1~3-5를 셀별로 돌리는 대신, 아래 한 줄로 정석 순서를 일괄 실행할 수 있다.
> ```
> !bash scripts/run_full_pipeline.sh
> ```

## 4. MLflow 결과 확인

기록된 run의 핵심 지표를 표로 출력한다.

In [ ]:
import mlflow, pandas as pd
mlflow.set_tracking_uri("mlruns")
exp = mlflow.get_experiment_by_name("thisabled-module1")
if exp is None:
    print("아직 기록된 실험이 없습니다 — 위 단계를 먼저 실행하세요.")
else:
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    cols = [c for c in runs.columns
            if c.startswith("metrics.") or c in ("tags.mlflow.runName", "start_time")]
    show = runs[cols].sort_values("start_time", ascending=False)
    pd.set_option("display.max_columns", None, "display.width", 200)
    display(show.head(20))
    print("\nKPI 대비: 긴급 Recall≥0.80, Macro-F1≥0.75 (긴급 클래스 포함 4-class 채점 기준)")